In [94]:
import torch
import os
import sys
import time
import random
import numpy as np

In [ ]:
ckpt_root = "/home/holywater2/25DFT/QHFlow/ckpts_old"
dataset_type = "md17"
dataset_names = ["water", "ethanol", "malondialdehyde", "uracil"]
# dataset_name = "water" #
dataset_name = dataset_names[0]
# ckpt_path = os.path.join(ckpt_root, "md17", "water", "checkpoints", "weights.ckpt")
ckpt_path = os.path.join(ckpt_root, dataset_type, dataset_name, "checkpoints", "weights.ckpt")

In [ ]:
ckpt_root = "/home/holywater2/25DFT/QHFlow/ckpts_old"
dataset_types = ["QH9Dynamic", "QH9Stable"]
dataset_type = dataset_types[0]
dataset_names = {
    "QH9Dynamic": ["geometry", "geometry-FT", "mol", "mol-FT"],
    "QH9Stable": ["random", "random-FT", "size_ood", "size_ood-FT"]
}
# dataset_name = "water" #
dataset_name = dataset_names[0]
# ckpt_path = os.path.join(ckpt_root, "md17", "water", "checkpoints", "weights.ckpt")
ckpt_path = os.path.join(ckpt_root, dataset_type, dataset_name, "checkpoints", "weights.ckpt")

In [96]:
loaded_ckpt = torch.load(ckpt_path)

/tmp/ipykernel_2975350/1438946806.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_ckpt = torch.load(ckpt_path)


In [97]:
def print_keys(loaded_ckpt):
    for idx, key in enumerate(loaded_ckpt.keys()):
        print(f"{idx}: {key}")

In [98]:
print_keys(loaded_ckpt)

0: epoch
1: global_step
2: pytorch-lightning_version
3: state_dict
4: loops
5: callbacks
6: optimizer_states
7: lr_schedulers
8: hparams_name
9: hyper_parameters


In [99]:
loaded_state_dict = loaded_ckpt["state_dict"]

In [100]:
print_keys(loaded_state_dict)

0: model.node_embedding.weight
1: model.distance_expansion._alpha
2: model.distance_expansion.cutoff
3: model.distance_expansion.logc
4: model.distance_expansion.n
5: model.distance_expansion.v
6: model.onebody_reduction.norm.tp.weight
7: model.onebody_reduction.norm.tp.output_mask
8: model.e3_gnn_layer.0.conv.tp_node.weight
9: model.e3_gnn_layer.0.conv.tp_node.output_mask
10: model.e3_gnn_layer.0.conv.fc_node.layer0.weight
11: model.e3_gnn_layer.0.conv.fc_node.layer1.weight
12: model.e3_gnn_layer.0.conv.layer_l0.layer0.weight
13: model.e3_gnn_layer.0.conv.layer_l0.layer1.weight
14: model.e3_gnn_layer.0.conv.linear_out.weight
15: model.e3_gnn_layer.0.conv.linear_out.bias
16: model.e3_gnn_layer.0.conv.linear_out.output_mask
17: model.e3_gnn_layer.0.conv.norm_gate.norm.tp.weight
18: model.e3_gnn_layer.0.conv.norm_gate.norm.tp.output_mask
19: model.e3_gnn_layer.0.conv.norm_gate.mul.weight
20: model.e3_gnn_layer.0.conv.norm_gate.mul.output_mask
21: model.e3_gnn_layer.0.conv.norm_gate.fc.0.

In [101]:
sys.path.append("/home/holywater2/25DFT/QHFlow/src")
# print(sys.path)
from models import QHFlow, get_default_model_args

In [102]:
conf = get_default_model_args(dataset_type)
conf["use_block_S"] = True
conf["use_block_H"] = False
print(conf)

{'in_node_features': 1, 'sh_lmax': 4, 'hidden_size': 128, 'bottle_hidden_size': 32, 'num_gnn_layers': 5, 'max_radius': 15, 'num_nodes': 10, 'radius_embed_dim': 16, 'max_T': 15, 'use_block_S': True, 'use_block_H': False, 'ham_dim': 24, 'ham_hidden': 288, 'dataset_type': 'md17'}


In [103]:
model = QHFlow(**conf)

/home/holywater2/miniforge3/envs/qhflow/lib/python3.12/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/holywater2/miniforge3/envs/qhflow/lib/python3.12/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/holywater2/miniforge3/envs/qhflow/lib/python3.12/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute

In [104]:
print_keys(model.state_dict())

0: node_embedding.weight
1: distance_expansion._alpha
2: distance_expansion.log_binomial_coeff
3: distance_expansion.n_indices
4: distance_expansion.v_indices
5: distance_expansion.cutoff
6: onebody_reduction.norm.tp.weight
7: onebody_reduction.norm.tp.output_mask
8: sigma_embedding.0.weight
9: sigma_embedding.0.bias
10: sigma_embedding.2.weight
11: sigma_embedding.2.bias
12: e3_gnn_layer.0.conv.tp_node.weight
13: e3_gnn_layer.0.conv.tp_node.output_mask
14: e3_gnn_layer.0.conv.fc_node.layer0.weight
15: e3_gnn_layer.0.conv.fc_node.layer1.weight
16: e3_gnn_layer.0.conv.layer_l0.layer0.weight
17: e3_gnn_layer.0.conv.layer_l0.layer1.weight
18: e3_gnn_layer.0.conv.linear_out.weight
19: e3_gnn_layer.0.conv.linear_out.bias
20: e3_gnn_layer.0.conv.linear_out.output_mask
21: e3_gnn_layer.0.conv.norm_gate.norm.tp.weight
22: e3_gnn_layer.0.conv.norm_gate.norm.tp.output_mask
23: e3_gnn_layer.0.conv.norm_gate.mul.weight
24: e3_gnn_layer.0.conv.norm_gate.mul.output_mask
25: e3_gnn_layer.0.conv.norm_

In [105]:
print_keys(loaded_state_dict)

0: model.node_embedding.weight
1: model.distance_expansion._alpha
2: model.distance_expansion.cutoff
3: model.distance_expansion.logc
4: model.distance_expansion.n
5: model.distance_expansion.v
6: model.onebody_reduction.norm.tp.weight
7: model.onebody_reduction.norm.tp.output_mask
8: model.e3_gnn_layer.0.conv.tp_node.weight
9: model.e3_gnn_layer.0.conv.tp_node.output_mask
10: model.e3_gnn_layer.0.conv.fc_node.layer0.weight
11: model.e3_gnn_layer.0.conv.fc_node.layer1.weight
12: model.e3_gnn_layer.0.conv.layer_l0.layer0.weight
13: model.e3_gnn_layer.0.conv.layer_l0.layer1.weight
14: model.e3_gnn_layer.0.conv.linear_out.weight
15: model.e3_gnn_layer.0.conv.linear_out.bias
16: model.e3_gnn_layer.0.conv.linear_out.output_mask
17: model.e3_gnn_layer.0.conv.norm_gate.norm.tp.weight
18: model.e3_gnn_layer.0.conv.norm_gate.norm.tp.output_mask
19: model.e3_gnn_layer.0.conv.norm_gate.mul.weight
20: model.e3_gnn_layer.0.conv.norm_gate.mul.output_mask
21: model.e3_gnn_layer.0.conv.norm_gate.fc.0.

In [106]:
# Missing key(s) in state_dict: "distance_expansion.log_binomial_coeff", "distance_expansion.n_indices", "distance_expansion.v_indices". 
# Unexpected key(s) in state_dict: "distance_expansion.logc", "distance_expansion.n", "distance_expansion.v". 
    
new_state_dict = dict()
for key in loaded_state_dict.keys():
    new_key = key.replace("model.", "")
    if new_key == "distance_expansion.logc":
        new_key = "distance_expansion.log_binomial_coeff"
    elif new_key == "distance_expansion.n":
        new_key = "distance_expansion.n_indices"
    elif new_key == "distance_expansion.v":
        new_key = "distance_expansion.v_indices"
    new_state_dict.update({new_key: loaded_state_dict[key]})

In [107]:
print_keys(new_state_dict)

0: node_embedding.weight
1: distance_expansion._alpha
2: distance_expansion.cutoff
3: distance_expansion.log_binomial_coeff
4: distance_expansion.n_indices
5: distance_expansion.v_indices
6: onebody_reduction.norm.tp.weight
7: onebody_reduction.norm.tp.output_mask
8: e3_gnn_layer.0.conv.tp_node.weight
9: e3_gnn_layer.0.conv.tp_node.output_mask
10: e3_gnn_layer.0.conv.fc_node.layer0.weight
11: e3_gnn_layer.0.conv.fc_node.layer1.weight
12: e3_gnn_layer.0.conv.layer_l0.layer0.weight
13: e3_gnn_layer.0.conv.layer_l0.layer1.weight
14: e3_gnn_layer.0.conv.linear_out.weight
15: e3_gnn_layer.0.conv.linear_out.bias
16: e3_gnn_layer.0.conv.linear_out.output_mask
17: e3_gnn_layer.0.conv.norm_gate.norm.tp.weight
18: e3_gnn_layer.0.conv.norm_gate.norm.tp.output_mask
19: e3_gnn_layer.0.conv.norm_gate.mul.weight
20: e3_gnn_layer.0.conv.norm_gate.mul.output_mask
21: e3_gnn_layer.0.conv.norm_gate.fc.0.weight
22: e3_gnn_layer.0.conv.norm_gate.fc.0.bias
23: e3_gnn_layer.0.conv.norm_gate.fc.2.weight
24: e

In [108]:
model.load_state_dict(state_dict=new_state_dict)

<All keys matched successfully>

In [109]:
torch.save(model.state_dict(), ckpt_path.replace(".ckpt", "_new.pt"))